# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL and contains mass spectrometry-derived protein data from human mast cell extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)# Access basic metadata informationmeta = dataset.metadataprint(f"{meta.name}: {meta.description}")print(f"Published on: {meta.datePublished}")print(f"Version: {meta.version}")

## 2. Data OverviewReview available record sets, fields, and their `@id`s.We list each available record set with its `@id`, and for each, enumerate its fields and columns with their `@id`s.

In [ ]:
# List all record sets in the dataset and their fields and columnsdef get_id(obj):    """Utility to safely fetch @id from an mlcroissant object for printing"""    return getattr(obj, '@id', None) or getattr(obj, 'id', None)record_sets = list(dataset.record_sets)print(f"Found {len(record_sets)} record sets:")for rs in record_sets:    print(f"- Record set name: {rs.name}, @id: {rs.id}")    if hasattr(rs, 'fields') and rs.fields:        print("  Fields:")        for f in rs.fields:            print(f"    {get_id(f)} ({f.name})")    if hasattr(rs, 'columns') and rs.columns:        print("  Columns:")        for c in rs.columns:            print(f"    {get_id(c)} ({c.name})")    print()

In [ ]:
# (Optional exploration) Print the first record from each record set, using `@id`for rs in record_sets:    rs_id = rs.id    print(f"\nFirst 1-2 records for record set @id: {rs_id}")    try:        for idx, record in enumerate(dataset.records(record_set=rs_id)):            print(record)            if idx >= 1:                break    except Exception as e:        print(f"Could not load records for {rs_id}: {e}")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis.> **Note**: Use the exact record set and field/column `@id` values from the overview above.

In [ ]:
# Extract all dataframes by record set `@id`# Collect all record set idsrecord_set_ids = [rs.id for rs in record_sets]dataframes = {}for record_set_id in record_set_ids:    try:        records = list(dataset.records(record_set=record_set_id))        df = pd.DataFrame(records)        dataframes[record_set_id] = df        print(f"Loaded DataFrame for RecordSet {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")    except Exception as e:        print(f"Could not load DataFrame for {record_set_id}: {e}")# Choose the first record set for demonstrationif record_set_ids:    rs0 = record_set_ids[0]    print(f"\nColumns in DataFrame for record set {rs0}:")    print(dataframes[rs0].columns.tolist())    display(dataframes[rs0].head())

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping by attributes.> All processing is done referencing fields by their `@id` as obtained in the previous overview.

In [ ]:
import numpy as np# Use the first loaded record set as an examplerecord_set_id = rs0df = dataframes[record_set_id]# Find a numeric field for filtering/normalization# We'll automatically select the first float/integer column, or set it manually if needednumeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()if numeric_columns:    numeric_field = numeric_columns[0]else:    # Fallback, try to parse one if known. You may replace this column id if desired.    numeric_field = df.columns[0]print(f"Using numeric field: {numeric_field} (from columns: {df.columns.tolist()})")threshold = df[numeric_field].quantile(0.75) if np.issubdtype(df[numeric_field].dtype, np.number) else 10filtered_df = df[df[numeric_field] > threshold]print(f"Filtered records with {numeric_field} > {threshold}:")display(filtered_df.head())# Normalization (Z-score)filtered_df = filtered_df.copy()filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()print(f"Normalized {numeric_field} for filtered records:")display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())# Try grouping by a likely categorical field (e.g., 'description' or other if present)poss_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]if poss_group_fields:    group_field = poss_group_fields[0]    print(f"Grouping by: {group_field}")    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()    display(grouped_df.head())else:    print("No suitable group field found for grouping.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Histogram of the numeric fieldplt.figure(figsize=(8, 4))sns.histplot(df[numeric_field], kde=True, bins=30)plt.title(f'Distribution of {numeric_field}')plt.xlabel(numeric_field)plt.show()# If grouping was performed, show a barplotif 'group_field' in locals():    top_n = 10 if grouped_df.shape[0] > 10 else grouped_df.shape[0]    plt.figure(figsize=(10, 5))    sns.barplot(data=grouped_df.head(top_n), x=group_field, y=numeric_field)    plt.title(f'Average {numeric_field} by {group_field}')    plt.xticks(rotation=45)    plt.show()

## 6. ConclusionIn this notebook, we loaded and inspected the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.- We explored available record sets and their fields using explicit `@id` references.- We extracted data into DataFrames, selected numeric fields for filtering and normalization, and visualized distributions.- Further domain-specific analysis can be performed by customizing the steps above based on knowledge of experimental design or required data granularity.